# 02. xLSTM: Extended Long Short-Term Memory (sLSTM & mLSTM)

- **Category:** 05_architectures / 03_recurrent_ssm_and_memory
- **Lineage & Citations:** Maximilian Beck et al., 2024 (*xLSTM: Extended Long Short-Term Memory*, arXiv:2405.04517); Sepp Hochreiter & Jürgen Schmidhuber, 1997 (*Long Short-Term Memory*, Neural Computation).
- **Purpose:** Solves LSTM storage saturation and parallelization bottlenecks via stabilized exponential gating (sLSTM) and associative matrix memory (mLSTM).
- **Topic Connections:** `🔗 Upstream Prerequisites:` [RNN LSTM GRU and Attention Seq2seq](01_rnn_lstm_gru_and_attention_seq2seq.ipynb) $\to$ `🔗 Downstream Syntheses:` [Structured State Space Models S4 and Lru](03_structured_state_space_models_s4_and_lru.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> - **Ultra-Low-SRAM Edge MCUs (< 16KB):** mLSTM maintains $C_t \in \mathbb{R}^{D \times D}$, taking $4D^2$ bytes per head; use sLSTM or standard GRU on tiny microcontrollers.  
> - **Unstabilized Exponential Gates:** Never omit running max tracker $m_t$; raw $\exp(\tilde{i}_t)$ causes FP32 overflow to `+inf` within 5 steps.

## Mathematical Foundations & Governing Equations

### 1. sLSTM (Scalar LSTM with Stabilized Exponential Gating)
$$m_t = \max(\log\sigma(\tilde{f}_t) + m_{t-1}, \ \tilde{i}_t), \quad i_t = \exp(\tilde{i}_t - m_t), \quad f_t = \sigma(\tilde{f}_t)\exp(m_{t-1} - m_t)$$
$$c_t = f_t c_{t-1} + i_t \tanh(\tilde{z}_t), \quad n_t = f_t n_{t-1} + i_t, \quad h_t = \sigma(o_t) \odot \frac{c_t}{n_t + \epsilon}$$

### 2. mLSTM (Matrix Associative Memory LSTM)
$$\mathbf{C}_t = f_t \mathbf{C}_{t-1} + i_t (\mathbf{v}_t \mathbf{k}_t^\top), \quad \mathbf{n}_t = f_t \mathbf{n}_{t-1} + i_t \mathbf{k}_t, \quad \tilde{\mathbf{h}}_t = \frac{\mathbf{C}_t \mathbf{q}_t}{\max(|\mathbf{n}_t^\top \mathbf{q}_t|, 1)}, \quad \mathbf{h}_t = \sigma(o_t) \odot \tilde{\mathbf{h}}_t$$

### Asymptotic Time & Memory Complexity
- **sLSTM Step Cost:** $\mathcal{O}(D^2)$ FLOPs, $\mathcal{O}(D)$ state memory.
- **mLSTM Step Cost:** $\mathcal{O}(D^2)$ FLOPs, $\mathcal{O}(D^2)$ state memory. $\mathcal{O}(1)$ step inference KV cache.

In [ ]:
import torch
import safetensors.torch
from pathlib import Path
import core

env = core.setup_env(seed=42)
CKPT_REL = Path("weights/05_architectures/03_recurrent_ssm_and_memory/02_xlstm_slstm_and_mlstm.safetensors")
ckpt_path = next(p / CKPT_REL for p in (Path.cwd(), *Path.cwd().parents) if (p / CKPT_REL).exists())
weights = safetensors.torch.load_file(ckpt_path)
x_seq = weights["x_seq"]
print(f"Loaded {len(weights)} golden tensors on {env.device}, sequence: {list(x_seq.shape)}")

In [ ]:
def slstm_forward(x_seq, w_i, w_f, w_z, w_o, u_i, u_f, u_z, u_o, b_i, b_f, b_z, b_o):
    B, T, _ = x_seq.shape
    D = w_i.shape[0]
    c_s, n_s, m_s, h_s = (torch.zeros(B, D, device=x_seq.device) for _ in range(4))
    outs = []
    for t in range(T):
        xt = x_seq[:, t, :]
        raw_i = xt @ w_i.t() + h_s @ u_i.t() + b_i
        raw_f = xt @ w_f.t() + h_s @ u_f.t() + b_f
        raw_z = xt @ w_z.t() + h_s @ u_z.t() + b_z
        raw_o = xt @ w_o.t() + h_s @ u_o.t() + b_o
        m_next = torch.maximum(torch.log(torch.sigmoid(raw_f) + 1e-12) + m_s, raw_i)
        i_gate = torch.exp(raw_i - m_next)
        f_gate = torch.sigmoid(raw_f) * torch.exp(m_s - m_next)
        c_s = f_gate * c_s + i_gate * torch.tanh(raw_z)
        n_s = f_gate * n_s + i_gate
        m_s = m_next
        h_s = torch.sigmoid(raw_o) * (c_s / (n_s + 1e-6))
        outs.append(h_s.unsqueeze(1))
    return torch.cat(outs, dim=1), c_s, n_s

def mlstm_forward(x_seq, w_q, w_k, w_v, w_i, w_f, w_o, b_q, b_k, b_v, b_i, b_f, b_o):
    B, T, _ = x_seq.shape
    D = w_q.shape[0]
    C_m, n_m, m_m = torch.zeros(B, D, D, device=x_seq.device), torch.zeros(B, D, device=x_seq.device), torch.zeros(B, 1, device=x_seq.device)
    outs = []
    for t in range(T):
        xt = x_seq[:, t, :]
        q_t, k_t, v_t = xt @ w_q.t() + b_q, xt @ w_k.t() + b_k, xt @ w_v.t() + b_v
        raw_i = (xt @ w_i.t() + b_i).mean(dim=-1, keepdim=True)
        raw_f = (xt @ w_f.t() + b_f).mean(dim=-1, keepdim=True)
        m_next = torch.maximum(torch.log(torch.sigmoid(raw_f) + 1e-12) + m_m, raw_i)
        i_gate, f_gate = torch.exp(raw_i - m_next), torch.sigmoid(raw_f) * torch.exp(m_m - m_next)
        C_m = f_gate.unsqueeze(-1) * C_m + i_gate.unsqueeze(-1) * torch.bmm(v_t.unsqueeze(-1), k_t.unsqueeze(1))
        n_m = f_gate * n_m + i_gate * k_t
        m_m = m_next
        C_q = torch.bmm(C_m, q_t.unsqueeze(-1)).squeeze(-1)
        denom = torch.clamp(torch.abs(torch.bmm(n_m.unsqueeze(1), q_t.unsqueeze(-1)).squeeze(-1)), min=1.0)
        outs.append((torch.sigmoid(xt @ w_o.t() + b_o) * (C_q / denom)).unsqueeze(1))
    return torch.cat(outs, dim=1), C_m, n_m

In [ ]:
# Quantitative Mathematical Assertions & Golden Parity Verification
s_seq, s_c, s_n = slstm_forward(
    x_seq, weights["slstm_w_i"], weights["slstm_w_f"], weights["slstm_w_z"], weights["slstm_w_o"],
    weights["slstm_u_i"], weights["slstm_u_f"], weights["slstm_u_z"], weights["slstm_u_o"],
    weights["slstm_b_i"], weights["slstm_b_f"], weights["slstm_b_z"], weights["slstm_b_o"]
)
assert float(torch.max(torch.abs(s_seq - weights["slstm_seq_out"]))) < 1e-6
assert float(torch.max(torch.abs(s_c - weights["slstm_final_c"]))) < 1e-6
assert float(torch.max(torch.abs(s_n - weights["slstm_final_n"]))) < 1e-6

m_seq, m_c, m_n = mlstm_forward(
    x_seq, weights["mlstm_w_q"], weights["mlstm_w_k"], weights["mlstm_w_v"],
    weights["mlstm_w_i"], weights["mlstm_w_f"], weights["mlstm_w_o"],
    weights["mlstm_b_q"], weights["mlstm_b_k"], weights["mlstm_b_v"],
    weights["mlstm_b_i"], weights["mlstm_b_f"], weights["mlstm_b_o"]
)
assert float(torch.max(torch.abs(m_seq - weights["mlstm_seq_out"]))) < 1e-6
assert float(torch.max(torch.abs(m_c - weights["mlstm_final_c"]))) < 1e-6
assert float(torch.max(torch.abs(m_n - weights["mlstm_final_n"]))) < 1e-6
print(f"Parity Verified: sLSTM={float(torch.max(torch.abs(s_seq - weights['slstm_seq_out']))):.2e}, mLSTM={float(torch.max(torch.abs(m_seq - weights['mlstm_seq_out']))):.2e}")

In [ ]:
import matplotlib.pyplot as plt
from core.benchmark import benchmark_fn, format_benchmark_table

b_s = benchmark_fn(lambda: slstm_forward(
    x_seq, weights["slstm_w_i"], weights["slstm_w_f"], weights["slstm_w_z"], weights["slstm_w_o"],
    weights["slstm_u_i"], weights["slstm_u_f"], weights["slstm_u_z"], weights["slstm_u_o"],
    weights["slstm_b_i"], weights["slstm_b_f"], weights["slstm_b_z"], weights["slstm_b_o"]
), iters=50, name="sLSTM Recurrent Unroll")
b_m = benchmark_fn(lambda: mlstm_forward(
    x_seq, weights["mlstm_w_q"], weights["mlstm_w_k"], weights["mlstm_w_v"],
    weights["mlstm_w_i"], weights["mlstm_w_f"], weights["mlstm_w_o"],
    weights["mlstm_b_q"], weights["mlstm_b_k"], weights["mlstm_b_v"],
    weights["mlstm_b_i"], weights["mlstm_b_f"], weights["mlstm_b_o"]
), iters=50, name="mLSTM Recurrent Unroll")
print(format_benchmark_table([b_s, b_m]))

fig, ax = plt.subplots(1, 3, figsize=(10, 2.6), dpi=100)
ax[0].imshow(s_seq[0].detach().numpy(), cmap="viridis", aspect="auto")
ax[0].set_title("sLSTM States [T, D]")
ax[1].imshow(m_seq[0].detach().numpy(), cmap="magma", aspect="auto")
ax[1].set_title("mLSTM States [T, D]")
ax[2].imshow(m_c[0].detach().numpy(), cmap="coolwarm", aspect="auto")
ax[2].set_title("mLSTM Matrix C_T [D, D]")
plt.tight_layout()
plt.show()

In [ ]:
# Pathological Misuse: Unstabilized Exponential Gating (Without Running Max Tracker m_t)
raw_inputs = torch.tensor([10.0, 30.0, 50.0, 95.0])
unstable_i = torch.exp(raw_inputs)
assert torch.isinf(unstable_i).any().item(), "Unstabilized exp gating must overflow!"

# Stabilized with running max m_t:
m_tracker = raw_inputs.max()
stabilized_i = torch.exp(raw_inputs - m_tracker)
assert not torch.isinf(stabilized_i).any().item()
print(f"Max-stabilized exponential gating verified: max={stabilized_i.max().item():.2e}, finite=True")

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Exponential Gating Overcomes Saturation:** sLSTM dynamically revises stored memories via stabilized exponential gating, giving it the ability to overwrite outdated states.
2. **Associative Matrix Memory Scales Capacity:** mLSTM's outer-product $v_t k_t^\top$ updates decouple capacity from sequence length while preserving $\mathcal{O}(1)$ step latency.
3. **Running Max Tracker Prevents Overflow:** Recursive tracker $m_t = \max(\log\sigma(f_t) + m_{t-1}, i_t)$ guarantees all exponential evaluations stay finite.

---
### Cross-Domain Generalisation Matrix
- **High-Frequency Financial Time Series:** mLSTM acts as an online associative covariance estimator, updating volatility matrices without storing historical ticks.
- **Real-Time Audio & Speech Synthesis:** sLSTM's exponential gating enables crisp onset detection and dynamic reverberation tracking at constant inference latency.
- **Robot Joint Impedance Control:** Associative matrix state stores instantaneous task jacobian mappings online during closed-loop manipulation.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [02_xlstm_slstm_and_mlstm.md](../../../markdown/05_architectures/03_recurrent_ssm_and_memory/02_xlstm_slstm_and_mlstm.md)
- **PyTorch 2.x Module:** [02_xlstm_slstm_and_mlstm.py](../../../pytorch2/05_architectures/03_recurrent_ssm_and_memory/02_xlstm_slstm_and_mlstm.py)
- **Functional JAX Module:** [02_xlstm_slstm_and_mlstm.py](../../../jax/05_architectures/03_recurrent_ssm_and_memory/02_xlstm_slstm_and_mlstm.py)
- **Native Rust Candle Track:** [02_xlstm_slstm_and_mlstm.rs](../../../candle/05_architectures/03_recurrent_ssm_and_memory/02_xlstm_slstm_and_mlstm.rs)
- **Hardware WGSL Kernel:** [02_xlstm_slstm_and_mlstm.rs](../../../kernels/05_architectures/03_recurrent_ssm_and_memory/02_xlstm_slstm_and_mlstm.rs)
- **Universal Golden SafeTensors:** [02_xlstm_slstm_and_mlstm.safetensors](../../../weights/05_architectures/03_recurrent_ssm_and_memory/02_xlstm_slstm_and_mlstm.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous Topic: 01. RNN LSTM GRU and Attention Seq2Seq](01_rnn_lstm_gru_and_attention_seq2seq.ipynb) | [Category Index](../../../README.md#05_architectures) | [Next Topic: 03. Structured State Space Models S4 and LRU](03_structured_state_space_models_s4_and_lru.ipynb) &rarr;